# Support Ticket Classification & Prioritization
This notebook covers the machine learning pipeline to automatically classify support tickets into categories and assign priority levels.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import joblib

## 1. Load Data
Loading the dataset from `data/customer_support_tickets.csv`.

In [ ]:
df = pd.read_csv('../data/customer_support_tickets.csv')
print("Dataset Shape:", df.shape)
df.head()

## 2. Data Cleaning & Preprocessing
Combine the Subject and Description to form the full text. Then, cleanly format the text input.

In [ ]:
# Fill NaNs if any
df['Ticket Subject'] = df['Ticket Subject'].fillna('')
df['Ticket Description'] = df['Ticket Description'].fillna('')
df['Ticket Type'] = df['Ticket Type'].fillna('General')
df['Ticket Priority'] = df['Ticket Priority'].fillna('Low')

# Combine Subject and Description for richer features
df['Full_Text'] = df['Ticket Subject'] + " " + df['Ticket Description']

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)  # Remove punctuation
    text = re.sub(r'\s+', ' ', text).strip() # Remove extra spaces
    return text

df['Clean_Text'] = df['Full_Text'].apply(clean_text)
print("Sample cleaned text:")
print(df['Clean_Text'].iloc[0])

## 3. Feature Extraction (TF-IDF)
Convert the textual data into a numerical matrix using Bag of Words / TF-IDF.

In [ ]:
X = df['Clean_Text']
y_category = df['Ticket Type']
y_priority = df['Ticket Priority']

# Train/Test splits for both targets
X_train, X_test, y_cat_train, y_cat_test = train_test_split(X, y_category, test_size=0.2, random_state=42)
_, _, y_prio_train, y_prio_test = train_test_split(X, y_priority, test_size=0.2, random_state=42)

vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

print("TF-IDF feature shape:", X_train_vec.shape)

## 4. Modeling: Ticket Categorization
Train a classifier to organize tickets into their appropriate types.

In [ ]:
categorizer = LogisticRegression(max_iter=1000)
categorizer.fit(X_train_vec, y_cat_train)

cat_preds = categorizer.predict(X_test_vec)
print("Category Accuracy:", accuracy_score(y_cat_test, cat_preds))
print("\nClassification Report (Category):\n")
print(classification_report(y_cat_test, cat_preds))

plt.figure(figsize=(10, 6))
sns.heatmap(confusion_matrix(y_cat_test, cat_preds), annot=True, fmt='d', 
            xticklabels=categorizer.classes_,
            yticklabels=categorizer.classes_, cmap="Blues")
plt.title('Confusion Matrix - Categories')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

## 5. Modeling: Priority Prediction
Train another model to assign an urgency level (High, Medium, Low).

In [ ]:
prioritizer = RandomForestClassifier(n_estimators=100, random_state=42)
prioritizer.fit(X_train_vec, y_prio_train)

prio_preds = prioritizer.predict(X_test_vec)
print("Priority Accuracy:", accuracy_score(y_prio_test, prio_preds))
print("\nClassification Report (Priority):\n")
print(classification_report(y_prio_test, prio_preds))

plt.figure(figsize=(8, 5))
sns.heatmap(confusion_matrix(y_prio_test, prio_preds), annot=True, fmt='d', 
            xticklabels=prioritizer.classes_,
            yticklabels=prioritizer.classes_, cmap="Oranges")
plt.title('Confusion Matrix - Priorities')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

## 6. Export Models
Save the vectorizer, the categorizer, and prioritizer models to disk so we can use them later in production apps.

In [ ]:
joblib.dump(vectorizer, '../models/tfidf_vectorizer.pkl')
joblib.dump(categorizer, '../models/ticket_categorizer.pkl')
joblib.dump(prioritizer, '../models/ticket_prioritizer.pkl')
print("Models saved successfully to the models/ directory.")